In [ ]:
import pandas as pd
import sectio
from tqdm import tqdm

all_families = sectio.catalog.list_families()
results = []

for family in tqdm(all_families, desc="Auditing Families"):
    if family == "DATA_DICTIONARY": 
        continue
    try:
        df_family = sectio.catalog.get_family(family)
        table_name = f"sections_{family.lower()}"
        
        for sid in df_family['Section_ID'].tolist():
            try:
                # 1. Compute high-precision properties
                cs = sectio.db.get_section(table_name, sid, subdivision='calc')
                
                # 2. Extract DB values from metadata
                # Using 0.0 as default to ensure they are floats
                db_a  = float(cs.metadata.get('A', 0.0))
                db_iy = float(cs.metadata.get('Iy', 0.0))
                db_iz = float(cs.metadata.get('Iz', 0.0))

                # 3. Store Raw Data
                results.append({
                    "Family": family,
                    "Section_ID": sid,
                    "DB_Area": db_a,
                    "Calc_Area": cs.area,
                    "DB_Iy": db_iy,
                    "Calc_Iy": cs.Iy,
                    "DB_Iz": db_iz,
                    "Calc_Iz": cs.Iz
                })
            except Exception:
                continue
    except Exception:
        continue

# Create the master DataFrame
audit_df = pd.DataFrame(results)

# Now you can calculate any error on the fly with Pandas:
# audit_df['Iy_Err'] = (abs(audit_df['Calc_Iy'] - audit_df['DB_Iy']) / audit_df['DB_Iy']) * 100

print(f"Audit complete! {len(audit_df)} rows captured.")
# display(audit_df.head())

In [ ]:
audit_df.sample(1).iloc[0]

In [ ]:
sectio.catalog.get_schema()

In [ ]:
print(sectio.catalog.get_schema('T'))